In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
results_df = spark.read.json("abfss://raw@databricksrg2026.dfs.core.windows.net/results.json")

In [0]:
results_df.printSchema()

define schema

In [0]:
results_schema  = "resultId INT, raceId INT, driverId INT, constructorId INT, number INT, grid INT, position INT, positionText STRING, positionOrder INT, points FLOAT, laps INT, time STRING, milliseconds INT, fastestLap INT, rank INT, fastestLapTime STRING, fastestLapSpeed FLOAT, statusId INT"


Transformattion

In [0]:
from pyspark.sql.functions import current_date

In [0]:
results_transform_df =  results_df.withColumnRenamed("resultId","result_id") \
                                      .withColumnRenamed("raceId","race_id") \
                                      .withColumnRenamed("driverId","driver_id") \
                                      .withColumnRenamed("constructorId","constructor_id")\
                                          .withColumnRenamed("positionText","position_text")\
                                           .withColumnRenamed("positionOrder","position_order")\
                                           .withColumnRenamed("fastestLap","fastest_lap")\
                                            .withColumnRenamed("fastestLapTime","fastest_lap_time")\
                                            .withColumnRenamed("fastestLapSpeed","fastest_lap_speed")\
                                                .withColumn("ingestion_date",current_date())
                                        

In [0]:
display(results_transform_df)

drop

In [0]:
results_final_df = results_transform_df.drop("statusId")

In [0]:
display(results_final_df)

write in parquet

In [0]:
results_final_df.write.mode("overwrite").parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/results")